![](01_agentic_introduction_planner%20-%20Jupyter%20Notebook_files/logo_datascientest_oCGZ.png)

* * *

#  Systèmes Agentiques ¶

##  Agent & Agent planner ¶

* * *

> Un agent est un système qui utilise un LLM pour décider du déroulé d'une tâche. Mais à mesure que l'application devient plus complexe, un agent unique montre vite ses limites :
> 
>   * L’agent peut **disposer de trop d’outils** : risque que le mauvais outil soit appelé, ou qu'il soit mal utilisé (mauvais argument, etc).
> 
>   * Le **contexte devient trop important** pour être géré efficacement par un seul agent.
> 
>   * **Trop de rôles à assumer** (planification, recherche, calcul, etc.) par l'agent.
> 
> 

> 
> Pour répondre à ces limites, une solution consiste à **découper l'application en plusieurs agents** (MAS : multi agent system), chacun étant plus simple, avec un rôle, et ses propres capacités de raisonnement. 
> 
> Les systèmes multi-agents offrent ainsi :
> 
>   * **Modularité** : chaque agent est indépendant, ce qui facilite le développement et les tests.
> 
>   * **Spécialisation** : chaque agent peut être conçu pour exceller dans un domaine précis, avec le contexte qui lui est associé.
> 
>   * **Contrôle** : on peut définir précisément les règles de communication entre les agents.
> 
> 


* * *

##  Architecture des systèmes multi-agents ¶

* * *

> Il existe plusieurs manières d'organiser un système multi-agents. Le choix dépendra de la **nature de la tâche** , de sa **complexité** , et du **besoin de spécialisation**.
> 
> ![](01_agentic_introduction_planner%20-%20Jupyter%20Notebook_files/mas_architectures_oCGZ.png)

## Single Agent¶

> Un seul agent gère à la fois le raisonnement et l’appel aux outils. Il prend la décision de bout en bout : quoi faire, dans quel ordre, avec quels paramètres.
> 
> ![](01_agentic_introduction_planner%20-%20Jupyter%20Notebook_files/mas_agent_structure_oCGZ.png) **Figure** : Un seul agent qui est capable de réfléchir, d'appeler des outils, d'ajouter des éléments dans sa mémoire, etc ([source](https://weaviate.io/assets/images/Components_of_an_AI_agent-2f1846374720471d6b11169203ccb865.png))
> 
> **Exemple** :
>
>> L’agent reçoit : _"Résume les emails reçus aujourd’hui et envoie-les à mon Slack"_
>> 
>> → Il appelle l’outil `gmail` avec `search_emails(query="after:2025/07/10 before:2025/07/11")`
>> 
>> → Il résume les emails via un module de réflexion
>> 
>> → Il appelle l’outil `slack` avec `send_message(channel="#inbox-summary", content=résumé)`
> 
> Ce système **convient parfaitement pour des tâches nécessitant pas un nombre d'actions trop important**.
> 
> ⚠️ Mais dès qu’il faut enchaîner **beaucoup d’actions** , le contexte devient vite important, et difficile à suivre : l'agent risque **d'oublier ce qu’il a déjà fait** , ou de perdre le fil.
> 
> **Exemples de limites** :
> 
>   * Une **recherche approfondie** (_"Détaille les différents programmes politiques des partis en France"_) peut nécessiter plusieurs recherches, synthèses intermédiaires, vérifications de sources...
> 
>   * Un **assistant de codage capable de créer une application complète** doit réfléchir à l’architecture, écrire plusieurs fichiers, tester le code... Le contexte risque d'exploser très rapidement, et de perdre rapidement le fil.
> 
> 


  * Compléter le code suivant pour la suite du cours.

  * Tester que les appels fonctionnent bien.


In [1]:
# ── Sanity check environnement local GPU ──────────────────────────────────
import platform, torch

print(f"Python       : {platform.python_version()}")
print(f"OS           : {platform.system()}")
print(f"CUDA dispo   : {torch.cuda.is_available()}  |  CUDA runtime : {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"Compute cap  : {torch.cuda.get_device_capability(0)}")
    x = torch.ones(3, 3).cuda()
    print(f"Test tenseur : OK {x.shape} sur {x.device}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device       : {device}")


Python       : 3.10.12
OS           : Linux
CUDA dispo   : True  |  CUDA runtime : 12.8
GPU          : NVIDIA GeForce RTX 5070
Compute cap  : (12, 0)
Test tenseur : OK torch.Size([3, 3]) sur cuda:0
device       : cuda


In [2]:
# ── Configuration LLM ─────────────────────────────────────────────────────
import os
from dotenv import load_dotenv, find_dotenv

# Charge LIORA_API_KEY (et autres clés) depuis .env — recherche vers la racine
_dotenv_path = find_dotenv()
if _dotenv_path:
    load_dotenv(_dotenv_path, override=False)
    print(f"✓ .env chargé : {_dotenv_path}")
else:
    print("⚠ .env introuvable — vérifier le répertoire de travail")

def build_llm():
    """Liora gateway (priorité) → Ollama local (fallback)."""
    if os.environ.get("LIORA_API_KEY"):
        from langchain_openai import ChatOpenAI
        print("✓ LLM : Liora gateway → gpt-4o-mini")
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0,
            api_key=os.environ["LIORA_API_KEY"],
            base_url="https://ai-gateway.liora.tech/",
        )
    # Fallback GPU locale — prérequis : ollama serve dans un terminal
    from langchain_ollama import ChatOllama
    print("⚠ LLM : Ollama llama3.1:8b (fallback local — LIORA_API_KEY absent)")
    return ChatOllama(model="llama3.1:8b", temperature=0)

llm = build_llm()


✓ .env chargé : /home/tchumi/NLP_cours/.env
✓ LLM : Liora gateway → gpt-4o-mini


In [6]:
from langchain_openai import ChatOpenAI
import os

# Choose the provider
# os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1" # If you use Groq
# os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"      # If you use OpenAI
os.environ["OPENAI_API_BASE"] = "https://ai-gateway.liora.tech/" # If you use the Liora GateWay

# Insert the api_key and model_name
api_key = os.environ["LIORA_API_KEY"]
model_name = "gpt-4o-mini"

llm = ChatOpenAI(model_name=model_name, temperature=0, api_key=api_key)
llm.invoke("Comment dire bonjours en 5 langues différentes")


AIMessage(content='Voici comment dire "bonjour" en cinq langues différentes :\n\n1. Anglais : Hello\n2. Espagnol : Hola\n3. Allemand : Guten Tag\n4. Italien : Ciao\n5. Chinois (mandarin) : 你好 (Nǐ hǎo)\n\nSi tu as besoin d\'autres traductions ou d\'informations, n\'hésite pas à demander !', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 16, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e1dafe7972', 'id': 'chatcmpl-Dz12cN8Ggp6Iu3Z53txpcOW47RJWx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f3cf9-0f14-7ac1-b036-ef3b24899c7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_toke

In [3]:
llm = ChatOpenAI(model_name=model_name, temperature=0, api_key=api_key)
llm.invoke("Comment dire bonjours en 5 langues différentes")


NameError: name 'ChatOpenAI' is not defined

> L’objectif est de créer un agent de recherche capable de sélectionner et d’utiliser automatiquement les outils les plus adaptés en fonction de la requête de l’utilisateur. Cet agent aura accès à trois outils :
> 
>   * `web_search` : effectuer une recherche sur Internet et récupérer des résultats récents.
> 
>   * `wikipedia` : obtenir des informations encyclopédiques sur un sujet.
> 
>   * `python_repl` : exécuter du code Python pour réaliser des calculs ou traiter des données.
> 
> 


  * Exécuter la cellule suivante pour définir nos trois outils.


In [7]:
from langchain.tools import tool
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia
from langchain_experimental.tools import PythonREPLTool


import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Recherche directement sur Interne et renvoie les 5 premiers résultats utiles."""
    
    url = f"https://html.duckduckgo.com/html/?q={quote_plus(query)}"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        r = requests.post(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        
        results = []
        for a in soup.select(".result__a")[:5]:
            link = a["href"]
            if "uddg=" in link:
                link = requests.utils.unquote(link.split("uddg=")[1].split("&")[0])
            title = a.get_text()
            snippet = a.find_next(class_="result__snippet")
            snippet = snippet.get_text() if snippet else ""
            
            results.append(f"{title}\n{link}\n{snippet[:200]}")
        
        return "\n\n---\n\n".join(results) or "Aucun résultat trouvé."
        
    except:
        return "Erreur lors de la recherche web (bloqué ou timeout)."




wikipedia.set_lang("fr")

wikipedia.wikipedia.USER_AGENT = (
    "my-langchain-app/0.1 "
    "(https://example.com; your-email@example.com)"
)


wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(lang="fr", doc_content_chars_max=10000)
)


def ensure_print(code: str) -> str:
    lines = code.strip().split("\n")
    if not lines:
        return code
    last = lines[-1].strip()
    if not last.startswith("print("):
        lines[-1] = f"print({last})"
    return "\n".join(lines)

@tool
def python_repl(code: str) -> str:
    """Exécute du code Python arbitraire. Le dernier statement est auto-print."""
    safe = ensure_print(code)
    repl = PythonREPLTool()
    return repl.run(safe)

tools = [web_search, wikipedia, python_repl]
tool_names = [t.name for t in tools]
tool_descriptions = {t.name: t.description for t in tools}
tools_text = "\n".join([f"- {name}: {desc}" for name, desc in tool_descriptions.items()])

print(tools_text)


- web_search: Recherche directement sur Interne et renvoie les 5 premiers résultats utiles.
- wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
- python_repl: Exécute du code Python arbitraire. Le dernier statement est auto-print.


  * Créez maintenant un agent de recherche capable de collecter, de vérifier et d'analyser des informations. 

  * Testez le système.


In [ ]:
from langchain.agents import create_agent
# Insérer voter code ici


In [8]:
from langchain.agents import create_agent


SYSTEM_PROMPT = f"""
Tu es un agent de recherche rigoureux capable de collecter,
de vérifier et d'analyser des informations.

Règles de fonctionnement :
- Utilise wikipedia pour obtenir un premier contexte encyclopédique.
- Utilise web_search pour rechercher des informations complémentaires
  ou vérifier des données susceptibles d'être récentes.
- Utilise python_repl lorsqu'un calcul, une comparaison numérique
  ou un traitement de données est nécessaire.
- Tu peux appeler plusieurs outils successivement.
- Recoupe les informations lorsqu'elles proviennent de plusieurs sources.
- N'invente jamais une information absente des résultats obtenus.
- Présente une réponse finale claire, structurée et synthétique.
"""


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


test_query = "Compare la population de la France et de l'Allemagne."


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": test_query,
            }
        ]
    }
)

final_message = result["messages"][-1]

print("\n--- RÉPONSE FINALE ---\n")
print(final_message.content)



--- RÉPONSE FINALE ---

Voici une comparaison de la population de la France et de l'Allemagne, basée sur les données les plus récentes :

### Population de la France
- **Population au 1er janvier 2025** : environ **68,8 millions** d'habitants.
- **Croissance** : La population a augmenté de 169 000 habitants par rapport à 2024, avec un solde migratoire positif de 152 000 habitants. La croissance annuelle est d'environ 0,3 % depuis 2017.
- **Fécondité** : Le taux de fécondité est de 1,68 en 2023, ce qui est supérieur à la moyenne européenne mais insuffisant pour le renouvellement des générations.

### Population de l'Allemagne
- **Population au 31 décembre 2023** : environ **83,86 millions** d'habitants.
- **Croissance** : L'Allemagne a connu un solde migratoire positif de 200 000 personnes, ce qui contribue à sa dynamique démographique. Le pays fait face à une dénatalité chronique et un vieillissement de la population.
- **Historique** : Ce chiffre représente un record dans l'histoire 

* * *

##  Système planificateur - Planification et Exécution ¶

* * *

> Pour éviter que l'agent se perde sur sa tâche de base, ou que le contexte de base devient trop important, l'approche par planification peut être une bonne option.

## Planification et Exécution¶

> Le système "Planification et Exécution" suit une approche en plusieurs étapes :
> 
>   1. **Planification** : Un premier **LLM** reçoit l’objectif global et élabore un **plan détaillé** sous forme de tâches simples et exécutables.
> 
>   2. **Exécution** : Un **agent** prend en charge ces tâches, les exécute séquentiellement (pour éviter que le contexte explose, il est possible d'exécuter chaque tâche complètement individuellement, c'est-à-dire dans le contexte des tâches précédentes) en utilisant des outils (par exemple : recherche sur Wikipedia, moteur web, calculs, etc.) et produit un **résultat intermédiaire**.
> 
>   3. **Validation / Replanification** : Une fois les étapes terminées, un dernier **appel LLM** vérifie si le résultat est pertinent et complet :
> 
> 

>
>>   * ✅ Si le résultat est satisfaisant : on retourne la réponse finale à l’utilisateur.
>> 
>>   * ❌ Sinon : on effectue une **replanification** avec le contexte capturé, puis l’agent recommence l’exécution.
>> 
>> 

> 
> ![](01_agentic_introduction_planner%20-%20Jupyter%20Notebook_files/mas_agent_plan_execute_oCGZ.png)
> 
> Ce système a l'avantage de décomposer la requête en étapes claires dans un premier temps, puis de la traiter par la suite. Vous trouverez son implémentation avec langchain et langgraph au travers de ce [lien](https://drive.google.com/file/d/1umE_DHHNAVlDR6cIqS_YRu3qt-T20Vcf/view?usp=sharing).

* * *

##  Système planificateur - REWOO ¶

* * *

> Le schéma _Plan → Exécute → Replanifie_ fonctionne bien tant que les tâches sont simples et peu nombreuses.
> 
> Mais dès que l’on décompose un objectif en plusieurs sous-tâches, deux limites apparaissent :
> 
>   * **Contexte trop lourd** : pour exécuter chaque étape, le LLM reçoit à nouveau tout l’historique du traitement des tâches précédentes : risque de perdre le modèle, coût plus élevé, et latence plus importante.
> 
>   * **Tâches trop isolées** : si on exécute les étapes sans contexte, certaines tâches échouent faute d’avoir les résultats précédents.
> 
> 

>
>> #### Exemple :¶
>> 
>> Un utilisateur demande : _"Peux-tu me dire si je dois arroser mes plantes aujourd’hui ?"_
>> 
>> Plan produit par un LLM :
>>     
>>     
>>     T1 : Récupérer les données météo du jour (pluie, humidité).
>>     T2 : Déterminer si un arrosage est nécessaire.
>>     
>> 
>> Le modèle **ne peut pas exécuter T2** sans avoir d’abord l’info météo obtenue en T1.
> 
> Donc, si on veut traiter les tâches individuellement, on a besoin d’un système qui **comprend les dépendances** entre tâches sans avoir besoin de donner tout l'historique. C’est l’objectif d'algorithme comme **REWOO** ou LLM compiler.

## Reasoning Without Observations¶

> REWOO propose de structurer le raisonnement sous forme de tâches liées entre elles, en identifiant explicitement leurs dépendances.
> 
> Le LLM produit :
> 
>   * une liste de tâches élémentaires.
> 
>   * les dépendances entre ces tâches.
> 
>   * une description concise de ce que chaque tâche doit produire.
> 
> 

> 
> Une fois cette structure générée, les tâches sont exécutées dans le bon ordre, mais **les résultats intermédiaires ne sont jamais renvoyés au LLM**. Le modèle ne raisonne qu'une seule fois au début, ce qui évite l’explosion du contexte.
> 
> ![](01_agentic_introduction_planner%20-%20Jupyter%20Notebook_files/mas_agent_rewoo_oCGZ.png)
> 
> Exemple : pour "Compare la politique énergétique de la France et de l’Allemagne", REWOO peut générer :
>     
>     
>     T1 : Collecter les informations sur la France.
>     T2 : Collecter les informations sur l’Allemagne.
>     T3 : Comparer les deux pays (dépend de T1 et T2).
>     T4 : Produire un résumé (dépend de T3).
>     
> 
> Le système exécute ensuite T1, T2, T3 et T4 en respectant les dépendances.

  * Exécuter le code suivant pour définir la partie planner.


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Literal, Optional


class PlanStep(BaseModel):
    plan: str = Field(
        ..., description="Detailed reasoning for this step"
    )
    tool: Optional[Literal[tuple(tool_names)]] = Field(
        ..., description="The tool to invoke at this step"
    )
    tool_input: Optional[str] = Field(
        ..., description="Input to provide to the selected tool"
    )
    evidence_dependencies: Optional[List[str]] = Field(
        default=None,
        description="List of evidence variable names (E1, E2, ...) required as inputs for this step. Optional."
    )
        
    evidence_var: str = Field(
        ..., description="Name of the variable that will store the resulting tool output (E1, E2, ...)"
    )


class PlansOutput(BaseModel):
    steps: List[PlanStep] = Field(
        ..., description="List of steps describing the full plan"
    )
        
        
        
llm = ChatOpenAI(model_name=model_name, temperature=0, api_key=api_key)
llm_plan = llm.with_structured_output(PlansOutput)


prompt_system = f"""
Your task is to construct a multi-step plan that solves the user’s task.

AVAILABLE TOOLS:
{tools_text}

Each step must describe:
- A detailed explanation of what the step accomplishes ("plan")
- A tool to call ({tool_names}) if needed
- The input for that tool ("tool_input") if needed
- A variable name storing the tool’s output ("evidence_var" such as E1, E2, E3…)

Some steps may depend on the results of previous steps.  
If a step requires output from earlier steps, you MUST list all required evidence variables in the field "evidence_dependencies".  
If the step does not depend on any previous output, set "evidence_dependencies" to null.

The output MUST strictly follow the structured output schema defined by the system.  
No text outside the structured schema is allowed.

Guidelines:
- Provide a tool call if needed.
- Use sequential variable names for evidence (E1, E2, E3…).
- Only include dependencies when they are actually needed.
- The final output must be valid according to the Pydantic model.
"""



# Définir le prompt
plan_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",  prompt_system),
        ("human", "Task: {task}"),
    ]
)

# Définir le modèle


# Chaîne avec les opérateurs modernes (pipe)
plan_chain = plan_prompt | llm_plan

complex_task = "Comparer les capitales de la France et de l’Allemagne et indiquer leurs populations respectives."

# Exemple d'appel
plan_output = plan_chain.invoke({"task": complex_task})
print(plan_output.model_dump_json(indent=4))


{
    "steps": [
        {
            "plan": "Je vais d'abord rechercher des informations sur la capitale de la France, qui est Paris, pour obtenir des détails sur sa population actuelle.",
            "tool": "web_search",
            "tool_input": "population de Paris 2023",
            "evidence_dependencies": null,
            "evidence_var": "E1"
        },
        {
            "plan": "Ensuite, je vais rechercher des informations sur la capitale de l'Allemagne, qui est Berlin, pour obtenir des détails sur sa population actuelle.",
            "tool": "web_search",
            "tool_input": "population de Berlin 2023",
            "evidence_dependencies": null,
            "evidence_var": "E2"
        },
        {
            "plan": "Après avoir obtenu les populations de Paris et Berlin, je vais comparer ces deux chiffres pour fournir une réponse complète sur les populations respectives des capitales de la France et de l'Allemagne.",
            "tool": "python_repl",
        

  * Ensuite, l'agent va traiter chaque étape avec uniquement le contexte délimité par le plan.


In [10]:
from pydantic import BaseModel, Field
# from langchain import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

class StepOutput(BaseModel):
    reasoning: str = Field(..., description="Le raisonnement complet pour traiter l'étape")
    answer: str = Field(..., description="La réponse finale, claire et exploitable")

step_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an agent responsible for solving ONE INTERMEDIATE STEP of a larger task.
You do NOT solve the full task — only the current step.

Here is all the data you must use:

──────── GLOBAL TASK ────────
{complex_task}

──────── TOOL ALREADY EXECUTED ────────
Name: {recommended_tool}
Input: {raw_tool_input}
Output: {tool_output}

──────── AVAILABLE DEPENDENCIES ────────
{dependencies}

──────── IMPORTANT RULES ────────
1. If the tool output is insufficient, inconsistent, unlikely, or clearly incorrect,
   you MUST call another tool to verify or refine the information.
2. You MUST NOT call the StepOutput tool UNTIL all necessary tool calls have been made.
   StepOutput is the FINAL ACTION that stops the process.
3. If after 2 attempts the tool still returns inconsistent or incorrect data, STOP trying.
   Produce your best possible answer based on the last tool output, clearly stating the uncertainty.
   
Now respond to the following instruction:
"""),
    ("user", "{messages}")
])

agent = create_agent(
    model=ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=api_key
    ),
    tools=tools,
    response_format=StepOutput
)

agent_with_prompt = step_agent_prompt | agent


step_result = agent_with_prompt.invoke({
    "messages": "quelle est la popultation de Paris ?",
    "complex_task": "Trouver des infos sur Paris.",
    "recommended_tool": "web_search",
    "raw_tool_input": "population de Paris",
    "dependencies": "",
    "tool_output": "154 de personnes"
})

step_result["structured_response"]


StepOutput(reasoning="Les résultats de la recherche indiquent que la population de Paris est de 2 103 778 habitants en 2023, selon plusieurs sources fiables, y compris Wikipedia et l'INSEE. Cela semble être une information cohérente et vérifiée.", answer='La population de Paris en 2023 est de 2 103 778 habitants.')

  * Maintenant que l'agent worker est défini, il est nécessaire d'implémenter la logique d'execution conditionnelle.


In [11]:
class Worker:
    def __init__(self, agent_with_prompt, tools):
        self.agent_with_prompt = agent_with_prompt   # ton agent déjà lié au prompt
        self.tools = tools                           # outils disponibles
    
    def _resolve_dependencies(self, tool_input, evidence):
        """Remplace E1, E2... dans tool_input."""
        if not tool_input:
            return tool_input
        resolved = tool_input
        for key, val in evidence.items():
            resolved = resolved.replace(key, val)
        return resolved

    def run(self, plan_output, complex_task):
        evidence = {}

        for step in plan_output.steps:
            print(f"\n\n===== EXECUTING {step.evidence_var} =====")

            # Résolution tool_input avec les dépendances
            resolved_input = self._resolve_dependencies(step.tool_input, evidence)

            # Exécution du tool si nécessaire
            tool_output = ""
            if step.tool:
                tool_fn = self.tools.get(step.tool)
                if tool_fn:
                    print(f"Running tool `{step.tool}` with input: {resolved_input}")
                    tool_output = tool_fn.run(resolved_input)
                else:
                    print(f"Tool `{step.tool}` not found !")

            # Construire les dépendances textuelles
            deps_text = ""
            if step.evidence_dependencies:
                deps_text = "\n".join(
                    f"{ev}: {evidence[ev]}"
                    for ev in step.evidence_dependencies
                )

            # Construire le message pour l’agent
            messages = f"{step.plan}\n\nTon objectif : {step.tool_input}"

            # Appel du step-agent
            result = self.agent_with_prompt.invoke({
                "messages": messages,
                "complex_task": complex_task,
                "recommended_tool": step.tool or "",
                "raw_tool_input": resolved_input or "",
                "dependencies": deps_text or "",
                "tool_output": tool_output or ""
            })

            structured = result["structured_response"]
            evidence[step.evidence_var] = structured.answer

            print(f"✔ Output for {step.evidence_var}: {structured.answer}")

        # Retour final
        final_key = list(evidence.keys())[-1]
        return evidence[final_key], evidence


worker = Worker(
    agent_with_prompt=agent_with_prompt,
    tools={
        "web_search": web_search,
        "wikipedia": wikipedia,
        "python_repl": python_repl
    }
)

workers_result, evidences = worker.run(
    plan_output=plan_output,        # ton plan du planner
    complex_task=complex_task
)




===== EXECUTING E1 =====
Running tool `web_search` with input: population de Paris 2023
✔ Output for E1: La population de Paris en 2023 est de 2 103 778 habitants.


===== EXECUTING E2 =====
Running tool `web_search` with input: population de Berlin 2023


Python REPL can execute arbitrary code. Use with caution.


✔ Output for E2: La population de Berlin en 2023 est d'environ 3,8 millions d'habitants.


===== EXECUTING E3 =====
Running tool `python_repl` with input: La population de Paris en 2023 est de 2 103 778 habitants., La population de Berlin en 2023 est d'environ 3,8 millions d'habitants.
✔ Output for E3: La population de Paris en 2023 est de 2 103 778 habitants, tandis que la population de Berlin est d'environ 3,8 millions d'habitants.


  * Et enfin, de répondre à la question à partir des résultats des workers.


In [12]:
class SolverOutput(BaseModel):
    answer: str = Field(..., description="La réponse finale au global_task")
        
solver_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are the FINAL SOLVER of a Reasoning-Without-Observations pipeline.

You receive:
- the original question
- the global_task
- the full plan
- the result produced by each worker step
- the final intermediate output (last evidence)

Your job:
- Use ALL plan steps and their worker outputs
- Understand the logical progression of the solution
- Solve the global task COMPLETELY
- DO NOT call any tools
- Produce ONLY a SolverOutput containing the final answer (no reasoning)

──────── QUESTION ────────
{complex_task}


──────── PLAN ────────
{plan_text}

──────── STEP RESULTS (EVIDENCE) ────────
{evidences}

Now produce ONLY the final answer as a SolverOutput.
"""),
    ("user", "Produce the final answer.")
])


solver_agent = create_agent(
    model=ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=api_key
    ),
    tools=[],  # aucune tool pour le solver
    response_format=SolverOutput
)

solver_agent_with_prompt = solver_prompt | solver_agent


class Solver:
    def __init__(self, solver_agent_with_prompt):
        self.solver_agent_with_prompt = solver_agent_with_prompt

    def run(self, complex_task, plan_output, evidences):
        
        # Construire le plan textuel
        plan_text = "\n".join(
            f"- Step {i+1}: {step.plan}"
            for i, step in enumerate(plan_output.steps)
        )

        # Construire les résultats des steps
        step_results = "\n".join(
            f"{ev}: {val}" for ev, val in evidences.items()
        )

        result = self.solver_agent_with_prompt.invoke({
            "complex_task": complex_task,
            "plan_text": plan_text,
            "evidences": step_results
        })

        return result["structured_response"].answer

solver = Solver(solver_agent_with_prompt)

solver.run(complex_task=complex_task, plan_output=plan_output, evidences=evidences)


"La population de Paris est de 2 103 778 habitants, tandis que la population de Berlin est d'environ 3,8 millions d'habitants."